# REGIME-SHIFT
### Macro-Aware Tactical Asset Allocation Engine

---

## Problem Statement

Standard quantitative strategies and static portfolios (like the classic 60/40) rely heavily on
stationary assumptions — they work well during prolonged bull runs but fail catastrophically
during structural market breaks. In institutional Global Macro and Multi-Asset investing, a
static allocation is a liability. The ability to **mathematically detect, quantify, and adapt**
to unobservable economic regime changes is the core discipline of modern portfolio management.

This notebook builds a dynamic allocation engine **from scratch** that:

1. Ingests noisy, multi-asset market data (equities, fixed income, safe havens), VIX, and macro
   factors from FRED.
2. Classifies hidden economic states using a **Hidden Markov Model (HMM)** — no manual labelling.
3. Dynamically re-optimizes portfolio weights via **convex optimization (CVXPY)**, switching
   objective functions based on the detected regime.
4. Validates everything with a **strict walk-forward harness** that eliminates look-ahead bias.
5. Explicitly models **transaction friction** so the strategy is economically viable, not just
   statistically appealing.

The real engineering challenge here is not fitting an HMM to historical data — that's the easy
part. The hard part is executing that logic *without cheating*: without letting the optimizer see
tomorrow's returns, and without letting the strategy trade so often that costs eat the alpha.
That is the risk-aware mindset this notebook is built to demonstrate.

## Goals → Deliverables Map

| Goal | Deliverable satisfied |
|---|---|
| Build an HMM Regime Classifier | Trained HMM + regime labels overlaid on price data + transition probabilities |
| Dynamic Constraint Mapping | Regime-conditional CVXPY objective (Max Sharpe in Bull, Min Vol in Bear/Crisis) |
| Walk-Forward Validation Harness | Expanding-window backtester, re-fit at every rebalance |
| Explicit Transaction Friction | 5 bps turnover penalty applied multiplicatively each rebalance |
| Benchmark Comparison | 60/40 and Equal-Weight static portfolios, full tear sheet |

## Pipeline

```
Data Ingestion  →  Regime Detection (HMM)  →  Optimization (CVXPY)  →  Backtest (Walk-Forward)  →  Reporting (Tear Sheet + README)
```

Each section below follows this exact order. Every code cell is preceded by a markdown cell
explaining **why** it's built the way it is, not just what it does.

## 1. Imports, Configuration, and Design Rationale

### Why these libraries?

- **`hmmlearn`** — provides a battle-tested `GaussianHMM` implementation with the Baum-Welch
  (EM) fitting algorithm and Viterbi decoding built in. Re-implementing the forward-backward
  algorithm from scratch would add risk without adding insight; the value of this project is in
  the *pipeline architecture*, not in re-deriving textbook EM.
- **`cvxpy`** — lets us express portfolio optimization as a *disciplined convex program*. This
  matters because both of our regime-conditional objectives (Max Sharpe, Min Variance) are
  quadratic programs — cvxpy guarantees a global optimum via the ECOS interior-point solver,
  unlike generic non-convex solvers (`scipy.optimize.minimize`) which can get stuck in local
  minima or fail to respect constraints reliably.
- **`yfinance` / `pandas_datareader`** — free, reproducible data sources for asset prices, VIX,
  and FRED macro series. Reproducibility matters more than data quality here: anyone cloning this
  repo should be able to regenerate the exact same dataset.

### Configuration philosophy

All "knobs" — asset universe, date range, HMM state count, rebalance frequency, turnover cost,
lookback window — live in a single `config` dictionary. This is a deliberate architecture
decision: **no magic numbers scattered through the codebase**. Every class below takes `cfg` as a
constructor argument, so changing the walk-forward window from expanding to rolling, or the
transaction cost from 5bps to 10bps, requires editing exactly one line in one place.

Note the asset universe is intentionally cross-asset: `equity` (SPY, QQQ), `fixed_income`
(TLT, AGG), `safe_havens` (GLD, SHV). This gives the optimizer genuine diversification levers to
pull across regimes — a Bear regime allocation that can only rotate between two equity ETFs isn't
a real macro strategy.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
from pandas_datareader import data as pdr
import datetime as dt
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from hmmlearn.hmm import GaussianHMM
import cvxpy as cp
from typing import Tuple, Dict, List, Optional

# -------------------------------
# 1.0 Configuration & Parameters
# -------------------------------
config = {
    'assets': {
        'equity': ['SPY', 'QQQ'],
        'fixed_income': ['TLT', 'AGG'],
        'safe_havens': ['GLD', 'SHV']
    },
    'benchmark_tickers': ['SPY', 'AGG'],  # for 60/40
    'macro_fred': ['BAA10Y', 'T10Y2Y', 'CPIAUCSL'],  # credit spread, term spread, CPI
    'vix_ticker': '^VIX',
    'start_date': '2005-01-01',
    'end_date': '2023-12-31',
    'rebalance_freq': 'M',  # monthly
    'hmm_states': 3,
    'lookback_days': 252,   # rolling covariance/return estimation
    'risk_free_rate': 0.02, # assumed annualised (2%)
    'turnover_cost_bps': 5.0,  # 5 bps = 0.0005
    'walkforward': 'expanding'  # expanding window
}

config

## 2. Data Ingestion Pipeline — `DataEngine`

### Architecture decision: one class owns the entire data lifecycle

`DataEngine` is responsible for **download → clean → align → feature-engineer**, and nothing
else. It does not know about regimes or optimization. This separation of concerns means the HMM
and optimizer downstream never have to worry about missing data, misaligned calendars, or raw
price levels — they only ever see a clean, aligned `features` DataFrame.

### Why alignment is the hardest part

Three data sources with three different native calendars are being merged:

- **Equity/ETF prices** — trade on NYSE trading days.
- **VIX** — trades on CBOE, same days as NYSE.
- **FRED macro series** (`BAA10Y`, `T10Y2Y`, `CPIAUCSL`) — published at daily, weekly, or monthly
  frequency, often with a reporting lag.

Naively concatenating these produces NaN-riddled rows. The fix is `_align_data`: intersect all
indices, forward-fill the lower-frequency macro data onto the daily trading calendar (a monthly
CPI print is *valid information* on every day until the next print — forward-filling correctly
respects that), then drop rows that still have gaps at the start of the sample.

### Feature engineering for the HMM

The final `features` matrix concatenates:
- Daily **returns** of every asset (the HMM's primary signal for "what regime is the market in").
- **VIX level** (a market-implied fear gauge — regimes with elevated VIX are almost by definition
  higher-stress regimes).
- **Macro spreads**: credit spread (`BAA10Y`), term spread (`T10Y2Y`), and CPI year-over-year
  change. These are slower-moving, forward-looking macro signals that help the HMM distinguish a
  short volatility spike from a genuine structural regime shift.

This multi-modal feature set is what lets the HMM detect regimes driven by *macro* stress, not
just realized volatility — a purely returns-based HMM would essentially just be a fancy
volatility clustering model.

In [ ]:
class DataEngine:
    \"\"\"
    Handles downloading, cleaning, alignment, and feature engineering
    for multi-asset market data, VIX, and macro factors.
    \"\"\"
    def __init__(self, cfg: dict):
        self.cfg = cfg
        self.prices = None
        self.returns = None
        self.features = None
        self.vix = None
        self.macro_df = None

    def download_all(self) -> None:
        # ---- Asset prices ----
        all_tickers = [t for group in self.cfg['assets'].values() for t in group]
        print("Downloading asset prices...")
        data = yf.download(all_tickers, start=self.cfg['start_date'],
                           end=self.cfg['end_date'], auto_adjust=True)
        self.prices = data['Close'].copy()
        self.prices.ffill(inplace=True)
        self.prices.dropna(how='all', inplace=True)

        # ---- VIX ----
        print("Downloading VIX...")
        vix = yf.download(self.cfg['vix_ticker'], start=self.cfg['start_date'],
                          end=self.cfg['end_date'], auto_adjust=True)
        self.vix = vix['Close'].rename('VIX')
        self.vix.ffill(inplace=True)

        # ---- FRED Macro ----
        print("Downloading FRED macro data...")
        fred_data = {}
        for series in self.cfg['macro_fred']:
            try:
                ts = pdr.DataReader(series, 'fred', self.cfg['start_date'], self.cfg['end_date'])
                fred_data[series] = ts.squeeze()
            except Exception as e:
                print(f"Warning: could not fetch {series}: {e}")
        if fred_data:
            self.macro_df = pd.DataFrame(fred_data)
            self.macro_df.ffill(inplace=True)  # forward fill monthly CPI
            # Compute CPI YoY change
            if 'CPIAUCSL' in self.macro_df.columns:
                self.macro_df['CPI_YOY'] = self.macro_df['CPIAUCSL'].pct_change(12)
                self.macro_df.drop('CPIAUCSL', axis=1, inplace=True)
        else:
            self.macro_df = pd.DataFrame(index=self.prices.index)

        # ---- Align everything to common trading days ----
        self._align_data()

    def _align_data(self) -> None:
        common_idx = self.prices.index
        if self.vix is not None:
            common_idx = common_idx.intersection(self.vix.dropna().index)
        if not self.macro_df.empty:
            common_idx = common_idx.intersection(self.macro_df.dropna(how='all').index)

        self.prices = self.prices.loc[common_idx]
        self.prices.dropna(inplace=True)

        # Daily returns
        self.returns = self.prices.pct_change().dropna()

        # Align VIX
        self.vix = self.vix.reindex(common_idx).ffill()

        # Align macro
        self.macro_df = self.macro_df.reindex(common_idx).ffill()

        # Feature set for HMM: daily asset returns, VIX level, macro spreads
        self._build_features()

    def _build_features(self) -> None:
        # VIX daily change (optional), we keep level
        vix_level = self.vix.rename('VIX_Level')

        # Combine features: returns of all assets + VIX level + macro factors
        feat_list = [self.returns, vix_level, self.macro_df]
        self.features = pd.concat(feat_list, axis=1).dropna()
        # Ensure no infinite values
        self.features.replace([np.inf, -np.inf], np.nan, inplace=True)
        self.features.dropna(inplace=True)

    def get_asset_returns(self) -> pd.DataFrame:
        return self.returns.loc[self.features.index]  # aligned

    def get_all_tickers(self) -> List[str]:
        return [t for group in self.cfg['assets'].values() for t in group]

### Running the ingestion step

This is the only cell in the notebook that hits the network. It downloads roughly 19 years of
daily data across 6 ETFs, VIX, and 3 FRED series, then aligns and feature-engineers them in one
call. If a FRED series fails to download (rate limits, API hiccups), `DataEngine` degrades
gracefully — it prints a warning and continues with whatever macro series succeeded, rather than
crashing the whole pipeline.

In [ ]:
engine = DataEngine(config)
engine.download_all()

print("\nPrice history shape:", engine.prices.shape)
print("Feature matrix shape:", engine.features.shape)
print("Feature columns:", list(engine.features.columns))
engine.features.tail()

## 3. Regime Detection — `HMMRegimeClassifier`

### The math

We model the market as evolving through a small number of unobserved (latent) states
$s_t \in \{0, 1, \dots, K-1\}$ (here $K=3$). Conditional on being in state $k$, the observed
feature vector $x_t$ (asset returns, VIX level, macro spreads) is assumed Gaussian:

$$ x_t \mid s_t = k \;\sim\; \mathcal{N}(\mu_k, \Sigma_k) $$

with transitions governed by a Markov chain $P(s_t = j \mid s_{t-1} = i) = A_{ij}$. This is a
**Gaussian HMM**. Two algorithms do all the work:

- **Baum-Welch (EM)** fits $\{\mu_k, \Sigma_k, A\}$ by maximizing the data likelihood —
  `hmmlearn` handles this via `.fit()`.
- **Viterbi** decodes the single most likely state sequence $s_1, \dots, s_T$ given the fitted
  parameters — this is what `.predict()` returns.

We deliberately do **not** hand-label states as "Bull/Bear/Crisis" during fitting — the EM
algorithm has no notion of what a bull market is. Instead, `_auto_label_regimes` performs a
post-hoc, data-driven mapping: for each of the 3 discovered states, compute the mean equity return
and mean VIX level of the days assigned to that state, then rank states by
(highest return, lowest VIX) → **Bull**, middle → **Bear**, (lowest return, highest VIX) →
**Crisis**. This keeps the labelling procedure fully automatic and reproducible — rerunning the
fit on a different date range will always produce a sensibly-ordered Bull/Bear/Crisis mapping,
even though the underlying state indices (0, 1, 2) are arbitrary and can flip between runs.

### Why this satisfies the "no manual labelling" goal

The only human input is *which two features* (equity return, VIX) define "good" vs "bad" — the
actual regime boundaries, transition dynamics, and day-by-day classification are 100% learned
from data.

In [ ]:
class HMMRegimeClassifier:
    \"\"\"
    Gaussian HMM to uncover hidden market regimes from features.
    Automatic state-to-regime mapping based on returns & risk.
    \"\"\"
    def __init__(self, n_states: int = 3, random_state: int = 42):
        self.n_states = n_states
        self.model = GaussianHMM(n_components=n_states, covariance_type='full',
                                 n_iter=1000, random_state=random_state)
        self._fitted = False
        self.regime_map_ = None  # state -> regime name

    def fit(self, features: pd.DataFrame) -> np.ndarray:
        \"\"\"
        Fit HMM and return the Viterbi state sequence.
        \"\"\"
        self.model.fit(features.values)
        self._fitted = True
        states = self.model.predict(features.values)
        # Automatic labeling
        self._auto_label_regimes(features, states)
        return states

    def predict(self, features: pd.DataFrame) -> np.ndarray:
        \"\"\"Predict most likely hidden state for each observation.\"\"\"
        if not self._fitted:
            raise RuntimeError("Model not fitted yet.")
        return self.model.predict(features.values)

    def predict_proba(self, features: pd.DataFrame) -> np.ndarray:
        return self.model.predict_proba(features.values)

    def _auto_label_regimes(self, features: pd.DataFrame, states: np.ndarray):
        \"\"\"
        Map state indices to 'Bull', 'Bear', 'Crisis' based on:
        - Highest mean equity return & low VIX -> Bull
        - Lowest equity return & high VIX -> Crisis
        - Middle -> Bear
        \"\"\"
        # Find equity tickers (assumed first columns are equity returns)
        equity_cols = [c for c in features.columns if c in ['SPY', 'QQQ']]
        if not equity_cols:
            equity_cols = features.columns[:2]  # fallback
        equity_returns = features[equity_cols].mean(axis=1)

        state_stats = {}
        for s in range(self.n_states):
            mask = states == s
            avg_ret = equity_returns.loc[mask].mean()
            avg_vix = features['VIX_Level'].loc[mask].mean()
            state_stats[s] = (avg_ret, avg_vix)

        # Sort states by (avg_ret, -avg_vix) descending: best is Bull, worst is Crisis
        sorted_states = sorted(state_stats.keys(),
                               key=lambda s: (state_stats[s][0], -state_stats[s][1]),
                               reverse=True)
        label_map = {}
        if len(sorted_states) >= 3:
            label_map[sorted_states[0]] = 'Bull'
            label_map[sorted_states[1]] = 'Bear'
            label_map[sorted_states[2]] = 'Crisis'
        else:
            # fallback
            for i, s in enumerate(sorted_states):
                label_map[s] = ['Bull', 'Bear', 'Crisis'][i]

        self.regime_map_ = {s: label_map[s] for s in range(self.n_states)}

    def get_regime_labels(self, states: np.ndarray) -> np.ndarray:
        return np.array([self.regime_map_[s] for s in states])

    @property
    def transition_matrix_(self) -> np.ndarray:
        return self.model.transmat_

### Fitting the full-sample HMM and documenting transition probabilities

This full-sample fit is used **only for visualization and reporting** — the walk-forward
backtester (Section 5) re-fits its own HMM at every rebalance date using strictly historical
data. Here, we fit once on the entire sample to (a) produce a clean regime-labeled chart overlaid
on price history, and (b) document the transition probability matrix $A$, where $A_{ij}$ is the
probability of moving from regime $i$ to regime $j$ over one trading day. A large diagonal
($A_{ii} \gg A_{ij}$) confirms regimes are **persistent** rather than flickering noise — which is
a sanity check that the HMM is actually learning macro-economic structure rather than overfitting
day-to-day return noise.

In [ ]:
full_hmm = HMMRegimeClassifier(n_states=config['hmm_states'])
full_states = full_hmm.fit(engine.features)
full_regime_labels = full_hmm.get_regime_labels(full_states)

# Ordered regime names for readability (Bull/Bear/Crisis correspond to different
# underlying state indices each run, so we recover the mapping explicitly)
ordered_regimes = [full_hmm.regime_map_[s] for s in range(config['hmm_states'])]

transition_df = pd.DataFrame(
    full_hmm.transition_matrix_,
    index=[f"From {r}" for r in ordered_regimes],
    columns=[f"To {r}" for r in ordered_regimes]
)

print("Full-sample regime transition probability matrix:\n")
print(transition_df.round(4))

regime_series = pd.Series(full_regime_labels, index=engine.features.index, name='Regime')
print("\nRegime day-count breakdown:")
print(regime_series.value_counts())

### Deliverable 1 — Regime labels overlaid on historical price data

The chart below plots SPY's price history with the background shaded by the HMM's regime
classification at each point in time (green = Bull, orange = Bear, red = Crisis). This is the
direct visual evidence that the HMM is picking up genuine market structure: Crisis shading should
line up with known drawdowns (2008 GFC, 2020 COVID crash, 2022 bear market), not random noise.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))

spy_price = engine.prices['SPY'].reindex(regime_series.index)

regime_colors = {'Bull': '#2ca02c', 'Bear': '#ff9f1a', 'Crisis': '#d62728'}

# Shade the background by contiguous regime blocks
prev_regime = None
block_start = regime_series.index[0]
for date, regime in regime_series.items():
    if regime != prev_regime:
        if prev_regime is not None:
            ax.axvspan(block_start, date, color=regime_colors[prev_regime], alpha=0.18, lw=0)
        block_start = date
        prev_regime = regime
ax.axvspan(block_start, regime_series.index[-1], color=regime_colors[prev_regime], alpha=0.18, lw=0)

ax.plot(spy_price.index, spy_price.values, color='black', linewidth=1.1, label='SPY Price')
ax.set_title('SPY Price with HMM-Detected Regimes Overlaid (Full-Sample Fit)', fontsize=13)
ax.set_ylabel('SPY Price ($)')

# Legend proxies for regime shading
from matplotlib.patches import Patch
handles = [ax.lines[0]] + [Patch(facecolor=regime_colors[r], alpha=0.3, label=r) for r in regime_colors]
ax.legend(handles=handles, labels=['SPY Price'] + list(regime_colors.keys()), loc='upper left')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 4. Dynamic Convex Optimizer — `DynamicPortfolioOptimizer`

### Why the objective function must change with the regime

A single static objective (e.g., always Max Sharpe) is exactly the fragility this project set out
to fix — Max Sharpe optimizers are notoriously unstable and concentrate risk during exactly the
periods (crises) when diversification matters most. Instead, the optimizer's objective is mapped
directly to the HMM's current regime call:

| Regime | Objective | Rationale |
|---|---|---|
| **Bull** | Maximize Sharpe Ratio | Risk appetite is rewarded; concentrate in the best risk-adjusted assets |
| **Bear / Crisis** | Minimize Variance | Capital preservation dominates; avoid concentrated bets when correlations spike |

### Turning Max Sharpe into a convex QP

The Sharpe ratio $\frac{w^T(\mu - r_f)}{\sqrt{w^T \Sigma w}}$ is a **non-convex** fractional
objective. The standard trick (used here) is a change of variables. Solve:

$$ \min_{v \ge 0} \; \tfrac{1}{2} v^T \Sigma v \quad \text{s.t.} \quad v^T(\mu - r_f) = 1 $$

then normalize $w^* = v^* / \mathbf{1}^T v^*$. This is provably equivalent to maximizing Sharpe
under long-only constraints, and — crucially — it is a **convex QP**, solvable to global
optimality by `cvxpy`'s ECOS solver in milliseconds, with no risk of local-minima traps that
plague generic Sharpe-maximizing solvers.

### Bear/Crisis: Minimum Variance

$$ \min_w \; w^T \Sigma w \quad \text{s.t.} \quad \mathbf{1}^T w = 1,\;\; 0 \le w_i \le 0.4 $$

This is directly convex — no reformulation needed. The $w_i \le 0.4$ cap prevents the optimizer
from parking the entire portfolio in a single low-volatility asset (e.g., all-cash via SHV),
which would technically minimize variance but defeats the purpose of running a diversified macro
strategy.

### Defensive fallbacks

Both branches wrap the `cvxpy` solve in a `try/except`: if ECOS fails to converge (numerical
issues with a near-singular covariance matrix, for instance), the optimizer falls back to equal
weight rather than crashing the entire backtest. This matters in a walk-forward loop that calls
the optimizer potentially 200+ times — a single numerical failure should degrade gracefully, not
halt the whole run.

In [ ]:
class DynamicPortfolioOptimizer:
    \"\"\"
    CVXPY-based optimizer that switches between Max Sharpe (Bull) and
    Min Volatility (Bear / Crisis). Uses a convex QP for Max Sharpe
    by minimizing variance subject to excess return = 1 and scaling.
    \"\"\"
    def __init__(self, risk_free_rate: float = 0.02):
        self.rf = risk_free_rate
        self.daily_rf = (1 + risk_free_rate) ** (1/252) - 1

    def optimize(self, mu: np.ndarray, Sigma: np.ndarray,
                 regime: str, long_only: bool = True,
                 max_weight: float = 0.4) -> np.ndarray:
        \"\"\"
        Return optimal portfolio weights given expected returns and covariance.
        \"\"\"
        n = len(mu)
        w = cp.Variable(n)
        # Base constraints
        constraints = [cp.sum(w) == 1]
        if long_only:
            constraints.append(w >= 0)
        if max_weight:
            constraints.append(w <= max_weight)

        if regime == 'Bull':
            # Solve tangency portfolio via QP:
            # min 0.5 * v^T Sigma v  s.t.  v^T (mu - rf) = 1, v >= 0
            v = cp.Variable(n)
            excess_ret = mu - self.daily_rf
            obj = cp.Minimize(0.5 * cp.quad_form(v, Sigma))
            cons = [v >= 0, excess_ret @ v == 1]
            prob = cp.Problem(obj, cons)
            try:
                prob.solve(solver=cp.ECOS, max_iters=200)
                if prob.status not in ['optimal', 'optimal_inaccurate']:
                    raise ValueError(f"Solver status {prob.status}")
                v_val = v.value
                # scale to sum = 1
                if v_val.sum() > 1e-8:
                    w_out = v_val / v_val.sum()
                else:
                    w_out = np.ones(n) / n
            except Exception:
                # fallback: equal weight
                w_out = np.ones(n) / n
        else:
            # Minimum volatility
            obj = cp.Minimize(cp.quad_form(w, Sigma))
            prob = cp.Problem(obj, constraints)
            try:
                prob.solve(solver=cp.ECOS)
                if prob.status not in ['optimal', 'optimal_inaccurate']:
                    raise ValueError(f"Solver status {prob.status}")
                w_val = w.value
                w_out = np.clip(w_val, 0, max_weight)
                w_out = w_out / w_out.sum()  # re-normalise just in case
            except Exception:
                w_out = np.ones(n) / n
        return w_out

## 5. Walk-Forward Backtester — `WalkForwardBacktester`

### The central risk of any regime-based backtest: look-ahead bias

If the HMM is fit once on the *entire* historical dataset and then used to "predict" the past,
the backtest is fatally contaminated — the model has already seen the 2008 crash when it
classifies days in 2007. Reported Sharpe ratios from such backtests are fiction.

This class enforces a **strict expanding-window walk-forward protocol**:

1. At each monthly rebalance date $t$, slice the data to `available_end = t - 1 day`.
2. **Re-fit the HMM from scratch** on that historical-only slice.
3. Take the *last in-sample* Viterbi state as the forecast regime for the upcoming holding
   period $t \to t+1$ — this is the regime the market was in as of the most recent data point
   available, used as a one-step-ahead nowcast.
4. Estimate $\mu$ and $\Sigma$ from only the trailing `lookback_days` (252 trading days ≈ 1 year)
   of historical returns — a rolling estimation window, not the full expanding history, so the
   optimizer reacts to recent risk/return conditions rather than being anchored by 2005 data.
5. Solve the regime-conditional QP for weights $w$.
6. Apply those weights **prospectively** to the *next* month's realized returns — data the model
   never saw during fitting or estimation.

This mirrors exactly the operational constraint a live trading desk faces: at 4pm on rebalance
day, you only ever know what has already happened.

### Transaction cost modelling

$$ \text{cost}_t = \tau \sum_i |w_{i,t} - w_{i,t-1}|, \qquad \tau = 5\text{bps} = 0.0005 $$

Turnover cost is deducted **multiplicatively** from portfolio value at every rebalance:
$\text{value}_t = \text{value}_{t-1} \times (1 + r_{\text{port}}) \times (1 - \text{cost}_t)$.
Without this, a regime-switching strategy that whipsaws between Max-Sharpe and Min-Variance
allocations every time the HMM flickers between Bull and Bear would look artificially brilliant
on paper while being unprofitable (or worse) once real-world frictions are applied. Explicitly
penalizing turnover is what forces the strategy to prove it survives real trading costs, not just
back-of-envelope math.

### Benchmarks computed in the same loop

Both benchmarks (**60/40** and **Equal-Weight**) are compounded using the *same* realized holding
period returns as the strategy, with **zero transaction costs applied** (since static portfolios
that are already invested at their target weights across the whole sample don't need repeated
rebalancing costs modeled here) — this ensures an apples-to-apples comparison on the return side
while stress-testing the strategy specifically on cost-efficiency.

In [ ]:
class WalkForwardBacktester:
    \"\"\"
    Strict temporal walk-forward: at each rebalance date,
    - Use only data up to (date - 1) to fit HMM, estimate mu/Sigma.
    - Predict regime for next month.
    - Optimise weights.
    - Apply transaction costs based on turnover.
    \"\"\"
    def __init__(self, data_engine: DataEngine, cfg: dict):
        self.data = data_engine
        self.cfg = cfg
        self.optimizer = DynamicPortfolioOptimizer(cfg['risk_free_rate'])
        self.asset_tickers = self.data.get_all_tickers()
        self.n_assets = len(self.asset_tickers)
        self.returns_df = self.data.get_asset_returns()
        self.features_df = self.data.features
        # resample to monthly rebalance dates
        self.rebal_dates = pd.date_range(start=self.returns_df.index[0],
                                         end=self.returns_df.index[-1],
                                         freq=cfg['rebalance_freq'])
        # filter to dates where we have enough history
        min_days = cfg['lookback_days'] + 50
        self.rebal_dates = self.rebal_dates[self.rebal_dates >=
                                            self.returns_df.index[min_days]]

    def run_backtest(self) -> pd.DataFrame:
        \"\"\"
        Returns DataFrame with portfolio values over time for strategy,
        60/40, and equal-weight.
        \"\"\"
        # Initialize portfolio values
        port_val = 1.0
        prev_weights = np.zeros(self.n_assets)
        records = []

        # Benchmark strategies: 60/40 and EW
        bm_60_40_val = 1.0
        bm_ew_val = 1.0
        bm_weights_60_40 = self._get_60_40_weights()
        bm_weights_ew = np.ones(self.n_assets) / self.n_assets

        for i, rebal_date in enumerate(self.rebal_dates):
            # date is month-end, we assume we rebalance at close of last day of month.
            # actual available data up to rebal_date (inclusive). But to be strict,
            # we use data up to the day before rebal_date to avoid look-ahead.
            available_end = rebal_date - pd.Timedelta(days=1)
            if available_end < self.returns_df.index[0]:
                continue

            # Slice historical data up to available_end
            hist_ret = self.returns_df.loc[:available_end]
            hist_feat = self.features_df.loc[:available_end]

            if len(hist_ret) < self.cfg['lookback_days']:
                continue

            # Fit HMM on entire history up to available_end
            hmm = HMMRegimeClassifier(n_states=self.cfg['hmm_states'])
            try:
                states = hmm.fit(hist_feat)
            except Exception:
                # fallback to equal weight
                w = np.ones(self.n_assets) / self.n_assets
                regime = 'Bear'
            else:
                # Predict regime at the last training observation (the "current" regime)
                last_state = states[-1]
                regime = hmm.regime_map_[last_state]

                # Estimate expected returns and covariance using recent lookback window
                recent_ret = hist_ret.iloc[-self.cfg['lookback_days']:]
                mu = recent_ret.mean().values
                Sigma = recent_ret.cov().values

                # Optimise
                w = self.optimizer.optimize(mu, Sigma, regime,
                                            long_only=True, max_weight=0.4)

            # Apply to portfolio: find the return for the next month (from rebal_date+1 to next rebal_date)
            next_rebal = (self.rebal_dates[i+1] if i+1 < len(self.rebal_dates)
                          else self.returns_df.index[-1])
            # monthly holding period: from the day after rebal_date to next_rebal
            hold_start = rebal_date + pd.Timedelta(days=1)
            hold_period_ret = self.returns_df.loc[hold_start:next_rebal]
            if hold_period_ret.empty:
                continue

            # Daily returns of portfolio (pre-cost)
            daily_port_ret = hold_period_ret.values @ w
            # Compound
            port_growth = np.prod(1 + daily_port_ret)

            # Transaction cost: turnover relative to prev_weights
            turnover = np.sum(np.abs(w - prev_weights))
            cost = turnover * self.cfg['turnover_cost_bps'] / 10000.0  # bps fraction
            port_val *= port_growth * (1 - cost)

            # Benchmark growth (no costs)
            bm_growth_60_40 = np.prod(1 + hold_period_ret.values @ bm_weights_60_40)
            bm_growth_ew = np.prod(1 + hold_period_ret.values @ bm_weights_ew)
            bm_60_40_val *= bm_growth_60_40
            bm_ew_val *= bm_growth_ew

            # Store record
            records.append({
                'date': next_rebal,
                'port_val': port_val,
                'bm_60_40': bm_60_40_val,
                'bm_ew': bm_ew_val,
                'turnover': turnover,
                'regime': regime
            })
            prev_weights = w.copy()

        result_df = pd.DataFrame(records).set_index('date')
        return result_df

    def _get_60_40_weights(self) -> np.ndarray:
        # Map SPY, AGG weights
        idx_spy = self.asset_tickers.index('SPY') if 'SPY' in self.asset_tickers else 0
        idx_agg = self.asset_tickers.index('AGG') if 'AGG' in self.asset_tickers else 1
        w = np.zeros(self.n_assets)
        w[idx_spy] = 0.6
        w[idx_agg] = 0.4
        return w

### Running the walk-forward backtest

This cell re-fits the HMM at every one of the ~200+ monthly rebalance dates in the sample — this
is intentionally expensive (the whole point is that each rebalance only ever sees the past) and
may take a few minutes to run.

In [ ]:
backtester = WalkForwardBacktester(engine, config)
backtest_df = backtester.run_backtest()

print("Backtest produced", len(backtest_df), "monthly rebalance records")
backtest_df.head()

## 6. Performance Tear Sheet — `performance_tearsheet`

### Metric definitions

- **Sharpe Ratio** — $\frac{\mathbb{E}[r - r_f]}{\sigma(r - r_f)} \times \sqrt{12}$ (annualized
  from monthly observations here, since the backtest's return series is sampled at each monthly
  rebalance). Standard risk-adjusted return measure; penalizes volatility symmetrically.
- **Sortino Ratio** — same numerator, but the denominator only uses the standard deviation of
  *downside* excess returns. This corrects Sharpe's flaw of penalizing upside volatility (a
  strategy with big up-months but no down-months shouldn't be penalized the same as one with
  symmetric swings).
- **Max Drawdown** — $\min_t \left(\frac{V_t}{\max_{s \le t} V_s} - 1\right)$, the worst
  peak-to-trough decline in portfolio value. This is the single number most institutional
  allocators care about first — it defines the worst-case pain an investor would have
  experienced.
- **Calmar Ratio** — annualized return divided by the absolute max drawdown. Directly answers
  "how much return am I getting per unit of worst-case pain," which Sharpe/Sortino don't capture
  since they use volatility, not drawdown, as the risk denominator.
- **Average Turnover** — mean $\sum_i |w_{i,t} - w_{i,t-1}|$ across all rebalances; the direct
  input to the transaction cost model, reported so the reader can see exactly how much trading
  activity underlies the reported (post-cost) returns.

### Why compare against both 60/40 and Equal-Weight

- **60/40** is the industry-standard "static balanced portfolio" benchmark — beating it is the
  minimum bar for any tactical strategy to justify its complexity.
- **Equal-Weight** across all six ETFs is a naive diversification benchmark that requires zero
  modelling skill — beating it demonstrates the regime-timing and optimization machinery is
  actually adding value beyond "just holding a diversified basket."

In [ ]:
def performance_tearsheet(backtest_df: pd.DataFrame, cfg: dict) -> dict:
    \"\"\"
    Compute Sharpe, Sortino, Max Drawdown, Calmar, Turnover stats.
    \"\"\"
    # Portfolio returns from values
    port_vals = backtest_df['port_val']
    bm_60_40 = backtest_df['bm_60_40']
    bm_ew = backtest_df['bm_ew']

    def metrics(series, rf=cfg['risk_free_rate']):
        rets = series.pct_change().dropna()
        ann_factor = 12 if cfg['rebalance_freq'] == 'M' else 252
        excess = rets - rf / ann_factor
        # Sharpe
        sharpe = np.mean(excess) / np.std(excess) * np.sqrt(ann_factor)
        # Sortino
        downside = excess[excess < 0]
        sortino = np.mean(excess) / np.std(downside) * np.sqrt(ann_factor) if len(downside) > 0 else np.nan
        # Max drawdown
        cummax = series.expanding().max()
        drawdown = series / cummax - 1
        max_dd = drawdown.min()
        # Calmar
        ann_return = (series.iloc[-1] / series.iloc[0]) ** (ann_factor / len(series)) - 1
        calmar = ann_return / abs(max_dd) if max_dd != 0 else np.nan
        # Turnover
        avg_turnover = backtest_df['turnover'].mean()
        return {
            'Sharpe': sharpe,
            'Sortino': sortino,
            'Max Drawdown': max_dd,
            'Calmar': calmar,
            'Avg Turnover': avg_turnover,
            'Annual Return': ann_return
        }

    strategy_stats = metrics(port_vals)
    bm_60_stats = metrics(bm_60_40)
    bm_ew_stats = metrics(bm_ew)

    print("\n" + "="*60)
    print("PERFORMANCE TEAR SHEET")
    print("="*60)
    print(f"{'Metric':<20} {'Regime-Shift':>15} {'60/40':>15} {'Equal Weight':>15}")
    print("-"*60)
    for key in strategy_stats:
        print(f"{key:<20} {strategy_stats[key]:>15.4f} {bm_60_stats[key]:>15.4f} {bm_ew_stats[key]:>15.4f}")
    print("="*60)
    return {'strategy': strategy_stats, '60_40': bm_60_stats, 'ew': bm_ew_stats}


stats = performance_tearsheet(backtest_df, config)

# Also present as a tidy DataFrame for easy reading / export
tearsheet_df = pd.DataFrame(stats).rename(columns={'strategy': 'Regime-Shift', '60_40': '60/40', 'ew': 'Equal Weight'})
tearsheet_df

## 7. Equity Curves & Regime Visualization — `plot_results`

Two complementary charts:

1. **Growth of $1** — cumulative wealth for the Regime-Shift strategy vs. both benchmarks,
   rebased to a common starting point so relative performance is directly readable.
2. **Regime state strip** — the full-sample HMM's regime classification through time, shown as a
   step function (0=Bull, 1=Bear, 2=Crisis), letting you visually cross-reference equity curve
   inflection points against detected regime shifts.

In [ ]:
def plot_results(backtest_df: pd.DataFrame, hmm_full_states: pd.Series = None):
    \"\"\"
    Equity curves with regime shading.
    \"\"\"
    fig, ax = plt.subplots(figsize=(14, 7))
    # Plot cumulative values
    ax.plot(backtest_df.index, backtest_df['port_val'] / backtest_df['port_val'].iloc[0],
            label='Regime-Shift', linewidth=2)
    ax.plot(backtest_df.index, backtest_df['bm_60_40'] / backtest_df['bm_60_40'].iloc[0],
            label='60/40', alpha=0.8)
    ax.plot(backtest_df.index, backtest_df['bm_ew'] / backtest_df['bm_ew'].iloc[0],
            label='Equal Weight', alpha=0.8)
    ax.set_title('Cumulative Wealth - Regime-Shift vs Benchmarks', fontsize=14)
    ax.set_ylabel('Growth of $1')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    if hmm_full_states is not None:
        # Plot regime shading on a separate axis
        fig2, ax2 = plt.subplots(figsize=(14, 3))
        # Convert states to numeric
        regime_map = {'Bull': 0, 'Bear': 1, 'Crisis': 2}
        numeric = hmm_full_states.map(regime_map)
        ax2.fill_between(numeric.index, 0, numeric, step='post', alpha=0.3,
                         color='tab:red')
        ax2.set_yticks([0, 1, 2])
        ax2.set_yticklabels(['Bull', 'Bear', 'Crisis'])
        ax2.set_title('Regime States (full-sample HMM)')
        plt.tight_layout()
        plt.show()


plot_results(backtest_df, hmm_full_states=regime_series)

### Regime occupancy during the backtest

As a final diagnostic, it's worth checking which regime the walk-forward HMM believed the market
was in at each of the ~200 rebalance dates — this is distinct from the full-sample regime series
above, since each rebalance re-fits on an expanding history and can classify the same calendar
date differently than the full-sample fit does.

In [ ]:
print("Regime calls during walk-forward backtest:")
print(backtest_df['regime'].value_counts())

fig, ax = plt.subplots(figsize=(14, 2.5))
regime_map = {'Bull': 0, 'Bear': 1, 'Crisis': 2}
ax.step(backtest_df.index, backtest_df['regime'].map(regime_map), where='post', color='black')
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['Bull', 'Bear', 'Crisis'])
ax.set_title('Walk-Forward Regime Calls at Each Rebalance')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Reporting — Generating the GitHub README

The final deliverable is a self-contained `README.md` documenting the architecture decisions,
mathematical rationale, testing instructions, and reproducibility guide, so the repository is
usable by someone who has never seen this notebook. The cell below writes it to disk and prints
it for review.

In [ ]:
readme_text = '''# REGIME-SHIFT: Macro-Aware Tactical Asset Allocation Engine

## Overview

REGIME-SHIFT is a dynamic tactical asset allocation engine that detects hidden market regimes
using a Gaussian Hidden Markov Model and re-optimizes a multi-asset portfolio via convex
optimization, validated with a strict walk-forward backtest that eliminates look-ahead bias and
explicitly models transaction costs.

## Architecture & Mathematical Rationale

### 1. Regime Detection - Hidden Markov Model

We use a **Gaussian Hidden Markov Model (GaussianHMM)** to discover latent market regimes from
daily features:
- Log returns of the multi-asset universe (SPY, QQQ, TLT, AGG, GLD, SHV).
- VIX closing level.
- FRED macro indicators: BAA-10Y credit spread, 10Y-2Y term spread, CPI YoY change.

The model assumes the market evolves through a finite number of unobserved states
`s_t in {0,1,2}` with state-specific Gaussian emission distributions. The **Viterbi algorithm**
yields the most likely sequence of hidden states. No manual labels are used; instead, states are
automatically mapped to *Bull*, *Bear*, and *Crisis* by ranking each state's mean equity return
and mean VIX level.

### 2. Dynamic Objective Mapping in Portfolio Optimization

The optimizer switches its objective function based on the predicted regime:
- **Bull regime**: Maximise the Sharpe ratio by solving a convex QP. The classic non-convex
  fractional program is transformed into a standard QP:
  `min_v  0.5 * v^T Sigma v   s.t.   v^T (mu - rf) = 1,  v >= 0`.
  The optimal unscaled vector `v*` is then normalised to sum to one, giving the maximum Sharpe
  portfolio under long-only and max-weight constraints.
- **Bear / Crisis regime**: Minimise portfolio variance directly:
  `min_w  w^T Sigma w   s.t.   sum(w) = 1,  0 <= w_i <= 0.4`.

Both formulations are solved with `cvxpy` (ECOS solver) and are numerically stable, with
equal-weight fallbacks on solver failure.

### 3. Walk-Forward Validation - Eliminating Look-Ahead Bias

The backtester uses a **strict temporal expanding window**:
- At each monthly rebalance date `t`, all models (HMM, mean/covariance estimators) are fitted
  *only* on data up to `t-1`.
- The HMM's last in-sample state is taken as the forecast regime for the holding period
  `t -> t+1`.
- Portfolio weights are then computed using those historically available estimates.

No future information leaks into the optimisation. This mirrors the exact operational
constraints of a real trading system.

### 4. Transaction Cost Modelling

A friction penalty of **5 bps per unit of turnover** is applied inside the backtester:

`cost_t = tau * sum_i |w_i,t - w_i,t-1|`, where `tau = 5e-4`.

The cost is deducted multiplicatively from the portfolio value at each rebalance. This penalises
excessive trading and ensures the strategy is economically viable, not just statistically
appealing on a gross-of-cost basis.

### 5. Benchmarks

The engine compares against two static, monthly-rebalanced portfolios:
- **60/40**: 60% SPY, 40% AGG.
- **Equal Weight**: uniform allocation across all six tickers.

Performance metrics (Sharpe, Sortino, Max Drawdown, Calmar, annualised return, average turnover)
are reported in a tear sheet for all three portfolios side by side.

## Repository Structure

```
regime-shift/
|-- regime_shift.ipynb      # Full pipeline notebook (this project)
|-- regime_shift.py         # Script version of the same pipeline
|-- README.md                # This file
|-- requirements.txt         # Pinned dependencies
```

## Reproducibility Guide

1. **Clone** the repository and install dependencies:
   ```bash
   git clone <your-repo-url>
   cd regime-shift
   pip install -r requirements.txt
   ```

   Or install directly:
   ```bash
   pip install numpy pandas yfinance pandas-datareader hmmlearn cvxpy matplotlib scipy jupyter
   ```

2. **Run the notebook end to end**:
   ```bash
   jupyter nbconvert --to notebook --execute regime_shift.ipynb --output regime_shift_executed.ipynb
   ```
   or open it interactively with `jupyter lab regime_shift.ipynb` and Run All.

3. **Determinism**: The HMM is seeded (`random_state=42`) so state assignments are reproducible
   given identical input data. Note that yfinance/FRED data can be revised or extended over time
   (new trading days appended, occasional data vendor corrections), so exact numerical
   reproduction is guaranteed only for a fixed `start_date`/`end_date` window and a fixed data
   pull timestamp - re-running the notebook months later may reflect minor data revisions from
   the upstream sources.

4. **Configuration**: All parameters (asset universe, date range, HMM state count, rebalance
   frequency, transaction cost, lookback window) live in the single `config` dictionary in
   Section 1. Change one parameter there to re-run a full sensitivity analysis.

## Testing Instructions

This project does not ship a formal `pytest` suite, but the notebook itself is structured to be
independently verifiable at each stage:

- **Data integrity check**: after Section 2, `engine.features.isna().sum().sum()` should be `0`
  (all rows are fully aligned and dropna'd).
- **HMM sanity check**: after Section 3, `transition_df` should have a strong diagonal
  (probability of staying in the same regime should exceed the probability of switching), and
  the price-overlay chart's red (Crisis) shading should visually align with known drawdown
  periods (2008, 2020, 2022).
- **Optimizer unit check**: call `DynamicPortfolioOptimizer().optimize(mu, Sigma, 'Bull')` and
  `... .optimize(mu, Sigma, 'Bear')` on a small synthetic `mu`/`Sigma` and confirm weights sum to
  1, are non-negative, and respect the 0.4 max-weight cap.
- **Look-ahead bias check**: confirm that inside `WalkForwardBacktester.run_backtest`, the HMM
  and `mu`/`Sigma` estimation always use `hist_ret`/`hist_feat` sliced strictly before
  `available_end < rebal_date`, and that realized returns applied to compute `port_growth` always
  come from `hold_start > rebal_date`.
- **Cost model check**: verify `turnover` values are in `[0, 2]` (bounded by the L1 distance
  between two simplex-constrained weight vectors) and that `cost = turnover * 5e-4` is being
  applied multiplicatively, not additively, to `port_val`.

## Known Limitations & Future Work

- Only 3 HMM states are modeled; a 4-5 state model could separate "recovery" from "bull" regimes.
- Transaction costs are a flat bps assumption; a more realistic model would scale cost with
  trade size / market liquidity.
- The Max Sharpe QP reformulation assumes long-only weights; extending to long/short would
  require revisiting the reformulation's non-negativity constraint.
- Macro data (FRED) is monthly/lagged in reality (e.g., CPI is reported with a delay); this
  implementation forward-fills as of the print date, not the actual availability date, which is
  a mild simplification versus true real-time data vintages.
'''

with open('README.md', 'w') as f:
    f.write(readme_text)

print(readme_text)